In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("subhajournal/busi-breast-ultrasound-images-dataset")

print("Path to dataset files:", path)

100%|██████████| 195M/195M [00:12<00:00, 16.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/subhajournal/busi-breast-ultrasound-images-dataset/versions/1


In [2]:
import os
import random
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
base_path = "/root/.cache/kagglehub/datasets/subhajournal/busi-breast-ultrasound-images-dataset/versions/1"

image_paths = []
labels = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        file_lower = file.lower()
        root_lower = root.lower()

        if file_lower.endswith(".png") and "mask" not in file_lower:
            if "benign" in root_lower:
                label = "benign"
            elif "malignant" in root_lower:
                label = "malignant"
            elif "normal" in root_lower:
                label = "normal"
            else:
                continue

            image_paths.append(os.path.join(root, file))
            labels.append(label)

df = pd.DataFrame({
    "image": image_paths,
    "label": labels
})

print("Total Images:", len(df))
print("\nOriginal Class Distribution:")
print(df["label"].value_counts())

Total Images: 780

Original Class Distribution:
label
benign       437
malignant    210
normal       133
Name: count, dtype: int64


In [5]:
label_map = {
    "normal": 0,
    "benign": 1,
    "malignant": 2
}

df["label"] = df["label"].map(label_map)

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain Class Distribution:")
print(train_df["label"].value_counts())

Train size: 563
Validation size: 100
Test size: 117

Train Class Distribution:
label
1    315
2    152
0     96
Name: count, dtype: int64


In [6]:
class BUSIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["image"]
        label = self.df.iloc[idx]["label"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [7]:
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(10, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [8]:
train_dataset = BUSIDataset(train_df, basic_transform)
val_dataset = BUSIDataset(val_df, basic_transform)
test_dataset = BUSIDataset(test_df, basic_transform)

train_dataset_aug = BUSIDataset(train_df, aug_transform)

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

In [9]:
class_counts = train_df["label"].value_counts().sort_index()
print("Train class counts:\n", class_counts)

class_weights = 1.0 / class_counts
sample_weights = train_df["label"].map(class_weights).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_over = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

train_loader_aug = DataLoader(
    train_dataset_aug,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=(device.type == "cuda")
)

Train class counts:
 label
0     96
1    315
2    152
Name: count, dtype: int64


In [10]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

In [11]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size=kernel_size, stride=stride,
            padding=padding, groups=in_ch, bias=False
        )
        self.dw_bn = nn.BatchNorm2d(in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.pw_bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.dw_bn(x)
        x = self.act(x)
        x = self.pointwise(x)
        x = self.pw_bn(x)
        x = self.act(x)
        return x

In [12]:
class PlainCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32),
            ConvBlock(32, 32),
            nn.MaxPool2d(2),

            ConvBlock(32, 64),
            ConvBlock(64, 64),
            nn.MaxPool2d(2),

            ConvBlock(64, 128),
            ConvBlock(128, 128),
            nn.MaxPool2d(2),

            ConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [13]:
class DSCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            DepthwiseSeparableConv(3, 32),
            DepthwiseSeparableConv(32, 32),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(32, 64),
            DepthwiseSeparableConv(64, 64),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(64, 128),
            DepthwiseSeparableConv(128, 128),
            nn.MaxPool2d(2),

            DepthwiseSeparableConv(128, 256),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [14]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction="none")

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()

In [15]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3, criterion=None):
    model = model.to(device)

    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_acc = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = val_correct / val_total
        scheduler.step(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

    model.load_state_dict(best_state)
    return model

In [16]:
def evaluate_model(model, test_loader, class_names=("Normal", "Benign", "Malignant")):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    print(classification_report(y_true, y_pred, target_names=list(class_names), zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return y_true, y_pred

In [19]:
print("\n====================")
print("BASELINE PLAIN CNN")
print("====================")

baseline_model = PlainCNN(num_classes=3)
baseline_model = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=25,
    lr=1e-3,
    criterion=nn.CrossEntropyLoss()
)

print("\nBaseline Evaluation")
baseline_true, baseline_pred = evaluate_model(baseline_model, test_loader)


BASELINE PLAIN CNN
Epoch 01/25 | Train Loss: 0.9742 | Train Acc: 0.5560 | Val Acc: 0.2700
Epoch 02/25 | Train Loss: 0.8972 | Train Acc: 0.6234 | Val Acc: 0.2900
Epoch 03/25 | Train Loss: 0.8651 | Train Acc: 0.6341 | Val Acc: 0.6900
Epoch 04/25 | Train Loss: 0.8177 | Train Acc: 0.6501 | Val Acc: 0.6000
Epoch 05/25 | Train Loss: 0.7883 | Train Acc: 0.6732 | Val Acc: 0.7100
Epoch 06/25 | Train Loss: 0.7520 | Train Acc: 0.6767 | Val Acc: 0.5900
Epoch 07/25 | Train Loss: 0.7573 | Train Acc: 0.6714 | Val Acc: 0.3700
Epoch 08/25 | Train Loss: 0.7288 | Train Acc: 0.6927 | Val Acc: 0.7300
Epoch 09/25 | Train Loss: 0.7316 | Train Acc: 0.6909 | Val Acc: 0.8200
Epoch 10/25 | Train Loss: 0.6845 | Train Acc: 0.7265 | Val Acc: 0.7400
Epoch 11/25 | Train Loss: 0.6684 | Train Acc: 0.7016 | Val Acc: 0.7400
Epoch 12/25 | Train Loss: 0.6605 | Train Acc: 0.7389 | Val Acc: 0.5800
Epoch 13/25 | Train Loss: 0.6322 | Train Acc: 0.7531 | Val Acc: 0.7800
Epoch 14/25 | Train Loss: 0.6035 | Train Acc: 0.7425 | Va

In [21]:
print("\n====================")
print("DEPTHWISE SEPARABLE CNN")
print("====================")

ds_model = DSCNN(num_classes=3)
ds_model = train_model(
    ds_model,
    train_loader,
    val_loader,
    epochs=25,
    lr=1e-3,
    criterion=nn.CrossEntropyLoss()
)

print("\nDepthwise Separable CNN Evaluation")
ds_true, ds_pred = evaluate_model(ds_model, test_loader)


DEPTHWISE SEPARABLE CNN
Epoch 01/25 | Train Loss: 1.0244 | Train Acc: 0.5364 | Val Acc: 0.5600
Epoch 02/25 | Train Loss: 0.9637 | Train Acc: 0.5631 | Val Acc: 0.5600
Epoch 03/25 | Train Loss: 0.9316 | Train Acc: 0.5737 | Val Acc: 0.6000
Epoch 04/25 | Train Loss: 0.8738 | Train Acc: 0.6306 | Val Acc: 0.5900
Epoch 05/25 | Train Loss: 0.8081 | Train Acc: 0.6465 | Val Acc: 0.7100
Epoch 06/25 | Train Loss: 0.7381 | Train Acc: 0.6679 | Val Acc: 0.7100
Epoch 07/25 | Train Loss: 0.6871 | Train Acc: 0.6909 | Val Acc: 0.7500
Epoch 08/25 | Train Loss: 0.6533 | Train Acc: 0.7105 | Val Acc: 0.8200
Epoch 09/25 | Train Loss: 0.5984 | Train Acc: 0.7531 | Val Acc: 0.7300
Epoch 10/25 | Train Loss: 0.5464 | Train Acc: 0.7886 | Val Acc: 0.7900
Epoch 11/25 | Train Loss: 0.4975 | Train Acc: 0.8064 | Val Acc: 0.6300
Epoch 12/25 | Train Loss: 0.4288 | Train Acc: 0.8366 | Val Acc: 0.7500
Epoch 13/25 | Train Loss: 0.3795 | Train Acc: 0.8668 | Val Acc: 0.8200
Epoch 14/25 | Train Loss: 0.3353 | Train Acc: 0.8739

In [23]:
print("\n====================")
print("DS-CNN + OVERSAMPLING")
print("====================")

ds_over_model = DSCNN(num_classes=3)
ds_over_model = train_model(
    ds_over_model,
    train_loader_over,
    val_loader,
    epochs=25,
    lr=1e-3,
    criterion=nn.CrossEntropyLoss()
)

print("\nOversampling Evaluation")
ds_over_true, ds_over_pred = evaluate_model(ds_over_model, test_loader)


DS-CNN + OVERSAMPLING
Epoch 01/25 | Train Loss: 1.0749 | Train Acc: 0.3872 | Val Acc: 0.2700
Epoch 02/25 | Train Loss: 1.0302 | Train Acc: 0.4902 | Val Acc: 0.2700
Epoch 03/25 | Train Loss: 0.9400 | Train Acc: 0.5773 | Val Acc: 0.4100
Epoch 04/25 | Train Loss: 0.8846 | Train Acc: 0.6146 | Val Acc: 0.5700
Epoch 05/25 | Train Loss: 0.7912 | Train Acc: 0.6892 | Val Acc: 0.7000
Epoch 06/25 | Train Loss: 0.6352 | Train Acc: 0.7531 | Val Acc: 0.7500
Epoch 07/25 | Train Loss: 0.5554 | Train Acc: 0.7567 | Val Acc: 0.4300
Epoch 08/25 | Train Loss: 0.4517 | Train Acc: 0.8401 | Val Acc: 0.6400
Epoch 09/25 | Train Loss: 0.5278 | Train Acc: 0.8028 | Val Acc: 0.7300
Epoch 10/25 | Train Loss: 0.4188 | Train Acc: 0.8472 | Val Acc: 0.7100
Epoch 11/25 | Train Loss: 0.2878 | Train Acc: 0.9094 | Val Acc: 0.7200
Epoch 12/25 | Train Loss: 0.3251 | Train Acc: 0.8917 | Val Acc: 0.7600
Epoch 13/25 | Train Loss: 0.2583 | Train Acc: 0.9094 | Val Acc: 0.7100
Epoch 14/25 | Train Loss: 0.2444 | Train Acc: 0.9183 |

In [24]:
print("\n====================")
print("DS-CNN + AUGMENTATION")
print("====================")

ds_aug_model = DSCNN(num_classes=3)
ds_aug_model = train_model(
    ds_aug_model,
    train_loader_aug,
    val_loader,
    epochs=25,
    lr=1e-3,
    criterion=nn.CrossEntropyLoss()
)

print("\nAugmentation Evaluation")
ds_aug_true, ds_aug_pred = evaluate_model(ds_aug_model, test_loader)


DS-CNN + AUGMENTATION
Epoch 01/25 | Train Loss: 0.9680 | Train Acc: 0.5364 | Val Acc: 0.2700
Epoch 02/25 | Train Loss: 0.9457 | Train Acc: 0.5861 | Val Acc: 0.2700
Epoch 03/25 | Train Loss: 0.9296 | Train Acc: 0.5879 | Val Acc: 0.4600
Epoch 04/25 | Train Loss: 0.9112 | Train Acc: 0.6075 | Val Acc: 0.6500
Epoch 05/25 | Train Loss: 0.8732 | Train Acc: 0.6234 | Val Acc: 0.6800
Epoch 06/25 | Train Loss: 0.8204 | Train Acc: 0.6430 | Val Acc: 0.5900
Epoch 07/25 | Train Loss: 0.7982 | Train Acc: 0.6483 | Val Acc: 0.7400
Epoch 08/25 | Train Loss: 0.7597 | Train Acc: 0.6909 | Val Acc: 0.7700
Epoch 09/25 | Train Loss: 0.7610 | Train Acc: 0.6963 | Val Acc: 0.7000
Epoch 10/25 | Train Loss: 0.7195 | Train Acc: 0.6856 | Val Acc: 0.7600
Epoch 11/25 | Train Loss: 0.7068 | Train Acc: 0.6945 | Val Acc: 0.7700
Epoch 12/25 | Train Loss: 0.6510 | Train Acc: 0.7496 | Val Acc: 0.7700
Epoch 13/25 | Train Loss: 0.6202 | Train Acc: 0.7371 | Val Acc: 0.8200
Epoch 14/25 | Train Loss: 0.6128 | Train Acc: 0.7655 |

In [26]:
print("\n====================")
print("DS-CNN + FOCAL LOSS")
print("====================")

ds_focal_model = DSCNN(num_classes=3)
ds_focal_model = train_model(
    ds_focal_model,
    train_loader,
    val_loader,
    epochs=25,
    lr=1e-3,
    criterion=FocalLoss(gamma=2.0)
)

print("\nFocal Loss Evaluation")
ds_focal_true, ds_focal_pred = evaluate_model(ds_focal_model, test_loader)


DS-CNN + FOCAL LOSS
Epoch 01/25 | Train Loss: 0.4240 | Train Acc: 0.5755 | Val Acc: 0.2700
Epoch 02/25 | Train Loss: 0.3873 | Train Acc: 0.5933 | Val Acc: 0.2700
Epoch 03/25 | Train Loss: 0.3493 | Train Acc: 0.6163 | Val Acc: 0.4400
Epoch 04/25 | Train Loss: 0.3222 | Train Acc: 0.6430 | Val Acc: 0.7800
Epoch 05/25 | Train Loss: 0.2838 | Train Acc: 0.6838 | Val Acc: 0.6400
Epoch 06/25 | Train Loss: 0.2659 | Train Acc: 0.7158 | Val Acc: 0.6100
Epoch 07/25 | Train Loss: 0.2502 | Train Acc: 0.7176 | Val Acc: 0.7400
Epoch 08/25 | Train Loss: 0.2177 | Train Acc: 0.7869 | Val Acc: 0.7400
Epoch 09/25 | Train Loss: 0.1887 | Train Acc: 0.7833 | Val Acc: 0.6600
Epoch 10/25 | Train Loss: 0.1724 | Train Acc: 0.8135 | Val Acc: 0.8200
Epoch 11/25 | Train Loss: 0.1645 | Train Acc: 0.8135 | Val Acc: 0.7700
Epoch 12/25 | Train Loss: 0.1578 | Train Acc: 0.8384 | Val Acc: 0.6300
Epoch 13/25 | Train Loss: 0.1129 | Train Acc: 0.8774 | Val Acc: 0.6900
Epoch 14/25 | Train Loss: 0.0998 | Train Acc: 0.8828 | V

In [27]:
results = pd.DataFrame({
    "Method": [
        "Baseline CNN",
        "Depthwise Separable CNN",
        "DS-CNN + Oversampling",
        "DS-CNN + Augmentation",
        "DS-CNN + Focal Loss"
    ],
    "Accuracy": [
        accuracy_score(baseline_true, baseline_pred),
        accuracy_score(ds_true, ds_pred),
        accuracy_score(ds_over_true, ds_over_pred),
        accuracy_score(ds_aug_true, ds_aug_pred),
        accuracy_score(ds_focal_true, ds_focal_pred)
    ],
    "F1 Score": [
        f1_score(baseline_true, baseline_pred, average="weighted"),
        f1_score(ds_true, ds_pred, average="weighted"),
        f1_score(ds_over_true, ds_over_pred, average="weighted"),
        f1_score(ds_aug_true, ds_aug_pred, average="weighted"),
        f1_score(ds_focal_true, ds_focal_pred, average="weighted")
    ]
})

print("\n====================")
print("FINAL COMPARISON")
print("====================")
print(results.sort_values(by="F1 Score", ascending=False).reset_index(drop=True))


FINAL COMPARISON
                    Method  Accuracy  F1 Score
0  Depthwise Separable CNN  0.803419  0.794706
1      DS-CNN + Focal Loss  0.777778  0.772280
2             Baseline CNN  0.743590  0.747296
3    DS-CNN + Oversampling  0.700855  0.704730
4    DS-CNN + Augmentation  0.700855  0.698524
